# Прекод

# Сборный проект-4

Вам поручено разработать демонстрационную версию поиска изображений по запросу.

Для демонстрационной версии нужно обучить модель, которая получит векторное представление изображения, векторное представление текста, а на выходе выдаст число от 0 до 1 — покажет, насколько текст и картинка подходят друг другу.

### Описание данных

Данные доступны по [ссылке](https://code.s3.yandex.net/datasets/dsplus_integrated_project_4.zip).

В файле `train_dataset.csv` находится информация, необходимая для обучения: имя файла изображения, идентификатор описания и текст описания. Для одной картинки может быть доступно до 5 описаний. Идентификатор описания имеет формат `<имя файла изображения>#<порядковый номер описания>`.

В папке `train_images` содержатся изображения для тренировки модели.

В файле `CrowdAnnotations.tsv` — данные по соответствию изображения и описания, полученные с помощью краудсорсинга. Номера колонок и соответствующий тип данных:

1. Имя файла изображения.
2. Идентификатор описания.
3. Доля людей, подтвердивших, что описание соответствует изображению.
4. Количество человек, подтвердивших, что описание соответствует изображению.
5. Количество человек, подтвердивших, что описание не соответствует изображению.

В файле `ExpertAnnotations.tsv` содержатся данные по соответствию изображения и описания, полученные в результате опроса экспертов. Номера колонок и соответствующий тип данных:

1. Имя файла изображения.
2. Идентификатор описания.

3, 4, 5 — оценки трёх экспертов.

Эксперты ставят оценки по шкале от 1 до 4, где 1 — изображение и запрос совершенно не соответствуют друг другу, 2 — запрос содержит элементы описания изображения, но в целом запрос тексту не соответствует, 3 — запрос и текст соответствуют с точностью до некоторых деталей, 4 — запрос и текст соответствуют полностью.

В файле `test_queries.csv` находится информация, необходимая для тестирования: идентификатор запроса, текст запроса и релевантное изображение. Для одной картинки может быть доступно до 5 описаний. Идентификатор описания имеет формат `<имя файла изображения>#<порядковый номер описания>`.

В папке `test_images` содержатся изображения для тестирования модели.

###  Загрузка библиотек

Использовалось в локальной версии:
torch: 2.0.1+cu118
torchvision: 0.15.2+cu118
tensorflow: 2.20.0
numpy: 1.26.4
CUDA torch: True
CUDA tf: []

In [ ]:
#!pip uninstall torch torchvision torchaudio -y
#!pip uninstall torch torchvision torchaudio -y  # повтор для надёжности

In [ ]:
#!pip cache purge

In [ ]:
#!pip list | grep torch

In [ ]:
#!pip install torch==2.4.0 torchvision==0.19.0 --index-url https://download.pytorch.org/whl/cu121

In [ ]:
#!python -c "import torch; print(torch.__version__); print(torch.cuda.is_available())"

In [5]:
import pandas  as pd
import matplotlib.pyplot as plt
import seaborn as sns

import re
import nltk

import os

import numpy as np
import tensorflow as tf

from tqdm.notebook import tqdm
from nltk.stem import WordNetLemmatizer
from nltk.corpus import stopwords, wordnet
from sklearn.feature_extraction.text import TfidfVectorizer

from tensorflow.keras.preprocessing.image import ImageDataGenerator, load_img, img_to_array
from PIL import Image

import torch
import torchvision
import torchvision.models as models
import torchvision.transforms as transforms

from sklearn.model_selection import GroupShuffleSplit
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import accuracy_score, roc_auc_score

In [6]:
print(f"torch: {torch.__version__}")
print(f"torchvision: {torchvision.__version__}")
print(f"tensorflow: {tf.__version__}")
print(f"numpy: {np.__version__}")
print(f"CUDA torch: {torch.cuda.is_available()}")
print(f"CUDA tf: {tf.config.experimental.list_physical_devices('GPU')}")

torch: 2.4.0+cu121
torchvision: 0.19.0+cu121
tensorflow: 2.7.0
numpy: 1.21.1
CUDA torch: False
CUDA tf: []


In [ ]:
pd.set_option('display.max_colwidth', None)

tqdm.pandas()

###  Загрузка данных

In [ ]:
CrowdAnn = pd.read_csv('/to_upload/CrowdAnnotations.tsv',
                    sep='\t', names=['image', 'query_id', '%_yes', 'yes', 'no'],)
CrowdAnn.info()

In [ ]:
CrowdAnn.head(50)

In [ ]:
ExpertAnn = pd.read_csv('/to_upload/ExpertAnnotations.tsv',
                    sep='\t', names=['image', 'query_id', 'expert_1', 'expert_2', 'expert_3'])
ExpertAnn.info()

In [ ]:
ExpertAnn.head(60)

In [ ]:
train_dataset = pd.read_csv('/to_upload/train_dataset.csv',
                    sep=',')
train_dataset.info()

In [ ]:
train_dataset.head(30)

In [ ]:
test_queries = pd.read_csv('/to_upload/test_queries.csv',
                    sep='|', index_col=0)
test_queries.info()

In [ ]:
test_queries.head(20)

Типы данных соответствуют описанию.

##  Исследовательский анализ данных

Наш датасет содержит экспертные и краудсорсинговые оценки соответствия текста и изображения.

В файле с экспертными мнениями для каждой пары изображение-текст имеются оценки от трёх специалистов. Для решения задачи вы должны эти оценки агрегировать — превратить в одну. Существует несколько способов агрегации оценок, самый простой — голосование большинства: за какую оценку проголосовала большая часть экспертов (в нашем случае 2 или 3), та оценка и ставится как итоговая. Поскольку число экспертов меньше числа классов, может случиться, что каждый эксперт поставит разные оценки, например: 1, 4, 2. В таком случае данную пару изображение-текст можно исключить из датасета.

Вы можете воспользоваться другим методом агрегации оценок или придумать свой.

В файле с краудсорсинговыми оценками информация расположена в таком порядке:

1. Доля исполнителей, подтвердивших, что текст **соответствует** картинке.
2. Количество исполнителей, подтвердивших, что текст **соответствует** картинке.
3. Количество исполнителей, подтвердивших, что текст **не соответствует** картинке.

После анализа экспертных и краудсорсинговых оценок выберите либо одну из них, либо объедините их в одну по какому-то критерию: например, оценка эксперта принимается с коэффициентом 0.6, а крауда — с коэффициентом 0.4.

Ваша модель должна возвращать на выходе вероятность соответствия изображения тексту, поэтому целевая переменная должна иметь значения от 0 до 1.


Проверим распределение оценок экспертов по всем предложенным изображениям

In [ ]:
plt.figure(figsize=(12, 6))
plt.hist([ExpertAnn['expert_1'], ExpertAnn['expert_2'], ExpertAnn['expert_3']], 
         bins=[0.5, 1.5, 2.5, 3.5, 4.5], 
         label=['expert_1', 'expert_2', 'expert_3'],
         edgecolor='black')
plt.legend()
plt.grid(True, alpha=0.3)
plt.xlabel('Значения')
plt.ylabel('Частота')
plt.title('Гистограмма распределения оценок экспертов')
plt.xticks([1, 2, 3, 4], fontsize=11)
plt.show()

Оценки экспертов на гистограмме показывают, что эксперт 1 склонен занижать оценку совпадения изображения запросу, в то время как эксперт 3 – несколько завышает оценку.

In [ ]:
plt.figure(figsize=(10, 6))
plt.hist(CrowdAnn['%_yes'], bins=[0, 0.25, 0.5, 0.75, 1.0], alpha=0.7, color='steelblue', edgecolor='black')
plt.xlabel('Значения')
plt.ylabel('Частота')
plt.title('Распределение оценок краудсорсинга')
plt.xticks([0, 0.33, 0.66, 1.0], ['0', '0.33', '0.66', '1'], fontsize=11)
plt.grid(True, alpha=0.3)
plt.show()

Большинство оценок людей негативные. Но есть значительная часть, где с результатом согласнен 1 из 3-х оценщиков.

Посмотрим, как соответствуют результаты оценки экспертов и краудсорсинга.

In [ ]:
merged = pd.merge(ExpertAnn, CrowdAnn, on=['image', 'query_id'], how='inner', suffixes=('_expert', '_crowd'))

In [ ]:
pivot_table = merged.groupby(['expert_1', 'expert_2', 'expert_3'])['%_yes'].mean().reset_index()

In [ ]:
pivot_table['expert_sum'] = pivot_table[['expert_1', 'expert_2', 'expert_3']].sum(axis=1)
print(pivot_table.sort_values('expert_sum'))

### Вывод

В сводной таблице при удовлетворении запроса 2-мя из 3-х экспертов и суммарной оценки в 8 баллов видим удвоение средней доли положительного ответа людей из краудсорсинга. Т.к. оценки экспертов имеют большую достоверность, а так же примем во внимание занижение оценки 1-ого эксперта. Можно сделать вывод, что значение в 8 баллов является пограничным состоянием удовлетворения запроса и значения от 8 баллов возьмем  за значение 1 – картинка подходит.

In [ ]:
ExpertAnn['expert_binary'] = (ExpertAnn[['expert_1', 'expert_2', 'expert_3']].sum(axis=1) >= 8).astype(int)

In [ ]:
ExpertAnn.head(50)

## 2. Проверка данных

В некоторых странах, где работает ваша компания, действуют ограничения по обработке изображений: поисковым сервисам и сервисам, предоставляющим возможность поиска, запрещено без разрешения родителей или законных представителей предоставлять любую информацию, в том числе, но не исключительно тексты, изображения, видео и аудио, содержащие описание, изображение или запись голоса детей. Ребёнком считается любой человек, не достигший 16 лет.

В вашем сервисе строго следуют законам стран, в которых работают. Поэтому при попытке посмотреть изображения, запрещённые законодательством, вместо картинок показывается дисклеймер:

> This image is unavailable in your country in compliance with local laws
>

Однако у вас в PoC нет возможности воспользоваться данным функционалом. Поэтому все изображения, которые нарушают данный закон, нужно удалить из обучающей выборки.


Лемматизируем текст для уменьшения вариаций слов в маске и первоначальной обработкой перед векторизацией текста.

In [ ]:
corpus = list(train_dataset['query_text'])

In [ ]:
def get_wordnet_pos(word):
    #Определяем POS-тег для WordNet
    tag = nltk.pos_tag([word])[0][1][0].upper()
    tag_dict = {
        "J": wordnet.ADJ,
        "N": wordnet.NOUN,
        "V": wordnet.VERB,
        "R": wordnet.ADV
    }
    return tag_dict.get(tag, wordnet.NOUN)

def lemmatize_english(text):
    #Лемматизация с учётом части речи
    lemmatizer = WordNetLemmatizer()
    words = text.split()
    lemmatized = []
    
    for word in words:
        pos = get_wordnet_pos(word)
        lemmatized.append(lemmatizer.lemmatize(word, pos))
    
    return ' '.join(lemmatized)

def clear_text(text):
    #Очистка текста
    res = re.sub(r"[^a-zA-Z']", ' ', text)
    c_text = " ".join(res.split())
    return c_text

def preprocess_text(text, remove_stopwords=False):
    #Полная предобработка
    text = text.lower()
    text = clear_text(text)
    text = lemmatize_english(text)
    
    if remove_stopwords:
        stop_words = set(stopwords.words('english'))
        words = text.split()
        words = [w for w in words if w not in stop_words]
        text = ' '.join(words)
    
    return text

# Проверка
print("Исходный текст:", corpus[15])
print()
print("Предобработанный (со стоп-словами):", preprocess_text(corpus[15], remove_stopwords=False))
print()
print("Предобработанный (без стоп-слов):", preprocess_text(corpus[15], remove_stopwords=True))

In [ ]:
sentence = "Beautiful flowers were growing in the big gardens. He went to the store for his mother"
result = preprocess_text(sentence, remove_stopwords=True)
print(result)
print("beautiful flower grow big garden go store mother")

In [ ]:
# Полученный результат применяем ко всем текстам
train_dataset['p_text'] = train_dataset['query_text'].progress_apply(preprocess_text, remove_stopwords=True)

In [ ]:
train_dataset

In [ ]:
# Паттерн для поиска возраста до 16 лет
age_pattern = r'\b(1[0-5]|[0-9])\s+(year|years)\s+old\b|\b(child|children|kid|kids|teen|teenager|underage|minor|boy|girl|young)\b'

mask = train_dataset['p_text'].str.contains(age_pattern, case=False, na=False)
train_dataset_clean = train_dataset[~mask].reset_index(drop=True)

print(f"Удалено запросов: {mask.sum()}")

In [ ]:
train_dataset_clean.info()

In [ ]:
train_dataset_clean.head(10)

In [ ]:
train = pd.merge(train_dataset_clean, ExpertAnn, on=['image', 'query_id'], how='left').drop(columns=['expert_1', 'expert_2', 'expert_3'])

In [ ]:
train.info()

In [ ]:
train

In [ ]:
train.duplicated().sum()

Все изображения, которые нарушают данный закон,  удалены - 1553 запросов. Обучающая выборка готова к дальнейшей работе.

##  Векторизация изображений

Перейдём к векторизации изображений.

Самый примитивный способ — прочесть изображение и превратить полученную матрицу в вектор. Такой способ нам не подходит: длина векторов может быть сильно разной, так как размеры изображений разные. Поэтому стоит обратиться к свёрточным сетям: они позволяют "выделить" главные компоненты изображений. Как это сделать? Нужно выбрать какую-либо архитектуру, например ResNet-18, посмотреть на слои и исключить полносвязные слои, которые отвечают за конечное предсказание. При этом можно загрузить модель данной архитектуры, предварительно натренированную на датасете ImageNet.

In [ ]:
# Загрузка предобученной модели ResNet-18
resnet18 = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)

# Удаляем последний полносвязный слой
resnet18 = torch.nn.Sequential(*list(resnet18.children())[:-1])
resnet18.eval()

# Трансформации для изображений
transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

In [ ]:
# Функция получения вектора
def get_image_vector(img_path):
    img = Image.open(img_path).convert('RGB')
    img_tensor = transform(img).unsqueeze(0)
    with torch.no_grad():
        features = resnet18(img_tensor)
    return features.squeeze().numpy()

# Векторизация в порядке train['image']
image_folder = '/to_upload/train_images'
vectors = []
missing_images = []

for img_name in train['image']:
    img_path = os.path.join(image_folder, img_name)
    if os.path.exists(img_path):
        vector = get_image_vector(img_path)
        vectors.append(vector)
    else:
        vectors.append(np.zeros(512))  # или None, или пропустить
        missing_images.append(img_name)
        print(f"Не найдено: {img_name}")

vectors = np.array(vectors)
print(f"Размер матрицы векторов: {vectors.shape}")  # (4269, 512)

## Векторизация текстов

Следующий этап — векторизация текстов. Вы можете поэкспериментировать с несколькими способами векторизации текстов:

- tf-idf
- word2vec
- \*трансформеры (например Bert)

\* — если вы изучали трансформеры в спринте Машинное обучение для текстов.


In [ ]:
# Векторизация текстов с помощью TF-IDF
tfidf = TfidfVectorizer(max_features=5000)  # можно ограничить количество признаков

# Обучаем векторайзер на текстах и преобразуем
X_text = tfidf.fit_transform(train['p_text'])

print(f"Размер матрицы: {X_text.shape}")  # (количество строк, количество признаков)
print(f"Количество уникальных слов: {len(tfidf.get_feature_names_out())}")

##  Объединение векторов

Подготовьте данные для обучения: объедините векторы изображений и векторы текстов с целевой переменной.

In [ ]:
# Преобразуем разреженную матрицу в numpy array
X_text_array = X_text.toarray()

# Объединяем с векторами изображений
X_combined = np.hstack([vectors, X_text_array])  # vectors из предыдущего задания
print(f"Размер объединенной матрицы: {X_combined.shape}")

##  Обучение модели предсказания соответствия

Для обучения разделите датасет на тренировочную и тестовую выборки. Простое случайное разбиение не подходит: нужно исключить попадание изображения и в обучающую, и в тестовую выборки.
Для того чтобы учесть изображения при разбиении, можно воспользоваться классом [GroupShuffleSplit](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.GroupShuffleSplit.html) из библиотеки sklearn.model_selection.

Код ниже разбивает датасет на тренировочную и тестовую выборки в пропорции 7:3 так, что строки с одинаковым значением 'group_column' будут содержаться либо в тестовом, либо в тренировочном датасете.

```
from sklearn.model_selection import GroupShuffleSplit
gss = GroupShuffleSplit(n_splits=1, train_size=.7, random_state=42)
train_indices, test_indices = next(gss.split(X=df.drop(columns=['target']), y=df['target'], groups=df['group_column']))
train_df, test_df = df.loc[train_indices], df.loc[test_indices]

```

Какую модель использовать — выберите самостоятельно. Также вам предстоит выбрать метрику качества либо реализовать свою.

In [ ]:
y = train['expert_binary']

gss = GroupShuffleSplit(n_splits=1, train_size=.7, random_state=42)

train_indices, test_indices = next(gss.split(X = X_combined, y = y, groups= train['image']))

X_train, X_test = X_combined[train_indices], X_combined[test_indices]
y_train, y_test = y.iloc[train_indices], y.iloc[test_indices]

Выберем RandomForestRegressor для работы с разреженными данными без масштабирования. Векторы изображений имеют значения от 0 до 1. TF-IDF векторы от 0 и более 1 . RandomForestRegressor устойчив к мультиколлинеарности, не требует нормализации распределения, эффективен при боьлшом количестве признаков.

In [ ]:
model = RandomForestRegressor(
    n_estimators=150,           # количество деревьев (больше = стабильнее)
    max_depth=30,               # глубина дерева (ограничиваем переобучение)
    min_samples_split=5,       # мин. объектов для разбиения узла
    min_samples_leaf=2,         # мин. объектов в листе
    max_features='sqrt',        # кол-во признаков для разбиения (sqrt(1461) ≈ 38)
    n_jobs=-1,                  
    random_state=42                             
)
model.fit(X_train, y_train)

Выберем 2 метрики: Accuracy - для простоты и наглядности, ROC-AUC - для оценки качества модели независимо от порога.

In [ ]:
y_pred = model.predict(X_test)

y_pred_binary = (y_pred > 0.5).astype(int)

accuracy = accuracy_score(y_test, y_pred_binary)
roc_auc = roc_auc_score(y_test, y_pred)

print(f"Accuracy:  {accuracy:.4f}")
print(f"ROC-AUC: {roc_auc:.4f}")

Результат работы RandomForestRegressor:
- Accuracy:  0.8325
- ROC-AUC: 0.7420

##  Тестирование модели

Настало время протестировать модель. Для этого получите эмбеддинги для всех тестовых изображений из папки `test_images`, выберите случайные 10 запросов из файла `test_queries.csv` и для каждого запроса выведите наиболее релевантное изображение. Сравните визуально качество поиска.

In [ ]:
# 1. ЗАГРУЗКА ТЕСТОВЫХ ДАННЫХ
image_folder = 'C:/Users/RabbitAl/Практикум/проекты/Сборный 5/test_images'


# 2. ЛЕММАТИЗАЦИЯ ТЕКСТОВ

if 'p_text' not in test_queries.columns:
    test_queries['p_text'] = test_queries['query_text'].apply(lambda x: preprocess_text(x, remove_stopwords=True))


# 3. ФИЛЬТРАЦИЯ: УДАЛЕНИЕ ЗАПРОСОВ ПРО НЕСОВЕРШЕННОЛЕТНИХ (ДО 16 ЛЕТ)

age_pattern = r'\b(1[0-5]|[0-9])\s+(year|years)\s+old\b|\b(child|children|kid|kids|teen|teenager|underage|minor|boy|girl|young)\b'

mask = test_queries['p_text'].str.contains(age_pattern, case=False, na=False)
removed_count = mask.sum()

print(f"Удалено запросов: {removed_count}")
test_queries_filtered = test_queries[~mask].reset_index(drop=True)
print(f"Осталось запросов: {len(test_queries_filtered)}")

# 4. ВЕКТОРИЗАЦИЯ ИЗОБРАЖЕНИЙ (только для отфильтрованных запросов)

unique_images = test_queries_filtered['image'].unique()
image_vectors = {}

for img_name in unique_images:
    img_path = os.path.join(image_folder, img_name)
    if os.path.exists(img_path):
        image_vectors[img_name] = get_image_vector(img_path)
    else:
        image_vectors[img_name] = np.zeros(512)
        print(f"Не найдено: {img_name}")

# 5. ПРЕДСКАЗАНИЕ ДЛЯ ОТФИЛЬТРОВАННЫХ ЗАПРОСОВ

predictions = []

for idx, row in test_queries_filtered.iterrows():
    text_vec = tfidf.transform([row['p_text']]).toarray()
    query_img_name = row['query_id'].split('#')[0]
    query_img_vector = image_vectors.get(query_img_name, np.zeros(512))
    X = np.hstack([query_img_vector.reshape(1, -1), text_vec])
    score = model.predict(X)[0]
    predictions.append(score)

test_queries_filtered['pred'] = predictions

# 6. ВИЗУАЛЬНОЕ СРАВНЕНИЕ ДЛЯ 10 СЛУЧАЙНЫХ ЗАПРОСОВ (без запрещенных)

sample_queries = test_queries_filtered.sample(min(10, len(test_queries_filtered)), random_state=42)

for idx, row in sample_queries.iterrows():
    query_img_name = row['query_id'].split('#')[0]
    score = row['pred']
    
    print(f"\n{'='*50}")
    print(f"Запрос: {row['query_text']}")
    print(f"Оценка соответствия: {score:.3f}")
    
    # Показываем две картинки рядом
    fig, axes = plt.subplots(1, 2, figsize=(8, 4))
    
    # Изображение-кандидат (query_id)
    query_img_path = os.path.join(image_folder, query_img_name)
    if os.path.exists(query_img_path):
        axes[0].imshow(Image.open(query_img_path))
        axes[0].set_title(f"Кандидат (query_id):\n{query_img_name}", fontsize=9)
    else:
        axes[0].text(0.5, 0.5, "Не найдено", ha='center')
    axes[0].axis('off')
    
    # Идеальное изображение (image)
    ideal_img_path = os.path.join(image_folder, row['image'])
    if os.path.exists(ideal_img_path):
        axes[1].imshow(Image.open(ideal_img_path))
        axes[1].set_title(f"Идеал (image):\n{row['image']}", fontsize=9)
    else:
        axes[1].text(0.5, 0.5, "Не найдено", ha='center')
    axes[1].axis('off')
    
    plt.suptitle(f"Score: {score:.3f}", fontsize=12)
    plt.tight_layout()
    plt.show()

##  Вывод

Модель RandomForestRegressor с параметрами  (max_depth=30, max_features='sqrt', min_samples_leaf=2, min_samples_split=5, n_estimators=150) хорошо показывает себя на обучающих данных: метрики качества
 - Accuracy: 83.25% — модель правильно классифицирует ~83% пар текст-изображение;
 - ROC-AUC: 0.742 — модель на 74% уверенно различает релевантные и нерелевантные пары.
 Но плохо показывает себя на тестовых, при этом собаки оцениваются выше — оценка вероятности 0.304, чем люди -  оценки 0.055-0.109. Возможно тестовые изображения/запросы отличаются от обучающих и ResNet + TF-IDF не улавливают связь запросов. Модель приемлема для демо-версии, но не готова к реальному использованию..
 



- [x]  Jupyter Notebook открыт
- [ ]  Весь код выполняется без ошибок
- [ ]  Ячейки с кодом расположены в порядке исполнения
- [ ]  Исследовательский анализ данных выполнен
- [ ]  Проверены экспертные оценки и краудсорсинговые оценки
- [ ]  Из датасета исключены те объекты, которые выходят за рамки юридических ограничений
- [ ]  Изображения векторизованы
- [ ]  Текстовые запросы векторизованы
- [ ]  Данные корректно разбиты на тренировочную и тестовую выборки
- [ ]  Предложена метрика качества работы модели
- [ ]  Предложена модель схожести изображений и текстового запроса
- [ ]  Модель обучена
- [ ]  По итогам обучения модели сделаны выводы
- [ ]  Проведено тестирование работы модели
- [ ]  По итогам тестирования визуально сравнили качество поиска